In [136]:
import torch

In [137]:
# Material Parameters
mu = 0.8
jm = 40.0
Identity = torch.eye(3)

In [138]:
# Synthetic Data Generation
def generate_synthetic_data(num_samples, lamda_min, lamda_max,eta_min,eta_max):
    lamda_train = torch.linspace(lamda_min, lamda_max, num_samples)
    I1_train = lamda_train**2 + 2/lamda_train
    P_true_train = gent_stress_function(lamda_train, mu, jm)   
    print(P_true_train)
    noise = add_noise(P_true_train,eta_min,eta_max,lamda_train)
    Y_train = P_true_train + noise
    print(Y_train)

     


In [139]:
# Gent Stress Function
def gent_stress_function(lamda, mu, jm):
    I1 = lamda**2 + 2/lamda
    stress = mu * (lamda - 1/lamda**2) / (1 - (I1 - 3)/jm)
    return stress

In [140]:
def deformation_gradient(lamda):
    diagonal_values = torch.stack([
        lamda,
        lamda**(-0.5),
        lamda**(-0.5)
    ], dim=-1)

    return torch.diag_embed(diagonal_values)

In [141]:
def frobenius_norm(F):
    return torch.sqrt(
        torch.sum((F - Identity)**2, dim=(-2, -1))
    )

In [142]:
def add_noise(P_true_train,eta_min,eta_max,lamda):
    P_char = torch.max(torch.abs(P_true_train))
    noise_sd_min = eta_min * P_char
    noise_sd_max = eta_max * P_char
    F = deformation_gradient(lamda)
    F_lambdamax = deformation_gradient(torch.max(lamda))
    q = 2.0
    t = frobenius_norm(F)
    ksy = torch.randn_like(P_true_train)
    


    conditional_noise_variance = torch.square(noise_sd_min) + (torch.square(noise_sd_max) - torch.square(noise_sd_min)) * ((t / frobenius_norm(F_lambdamax))**q)
    print(conditional_noise_variance)
    print(torch.max(torch.sqrt(conditional_noise_variance)))
    print(torch.min(torch.sqrt(conditional_noise_variance)))
    return torch.sqrt(conditional_noise_variance) * ksy




In [143]:
generate_synthetic_data(num_samples=50, lamda_min=1.0, lamda_max=4.0, eta_min=0.03,eta_max=0.005)


tensor([0.0000, 0.1387, 0.2633, 0.3768, 0.4816, 0.5794, 0.6716, 0.7592, 0.8432,
        0.9243, 1.0031, 1.0800, 1.1555, 1.2299, 1.3035, 1.3766, 1.4494, 1.5222,
        1.5951, 1.6683, 1.7419, 1.8162, 1.8912, 1.9671, 2.0440, 2.1221, 2.2015,
        2.2822, 2.3646, 2.4486, 2.5344, 2.6222, 2.7121, 2.8043, 2.8989, 2.9961,
        3.0961, 3.1990, 3.3051, 3.4145, 3.5275, 3.6444, 3.7654, 3.8908, 4.0209,
        4.1561, 4.2966, 4.4429, 4.5955, 4.7547])
tensor([0.0203, 0.0203, 0.0203, 0.0202, 0.0202, 0.0201, 0.0200, 0.0199, 0.0197,
        0.0196, 0.0194, 0.0192, 0.0190, 0.0188, 0.0185, 0.0183, 0.0180, 0.0177,
        0.0174, 0.0171, 0.0168, 0.0164, 0.0161, 0.0157, 0.0153, 0.0149, 0.0145,
        0.0140, 0.0136, 0.0131, 0.0126, 0.0121, 0.0116, 0.0111, 0.0106, 0.0100,
        0.0094, 0.0088, 0.0082, 0.0076, 0.0070, 0.0063, 0.0057, 0.0050, 0.0043,
        0.0036, 0.0028, 0.0021, 0.0013, 0.0006])
tensor(0.1426)
tensor(0.0238)
tensor([ 0.0242, -0.0665,  0.1474,  0.3928,  0.4486,  0.3744,  0.6701,  